In [1]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os, json, shutil, importlib
sys.path.append("/content/drive/MyDrive/tcc")
from src import config

import numpy as np
import pandas as pd
import torch

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NENHUMA — troque o ambiente para T4")

Mounted at /content/drive
GPU: Tesla T4


In [7]:
%%writefile /content/drive/MyDrive/tcc/src/modelos.py
import torch
import torch.nn as nn
from torchvision import models


class CNNArtigo(nn.Module):
    """CNN de Martin-Rodriguez et al. (2023), adaptada para entrada 256x256x3."""

    def __init__(self, canais=3, classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(canais, 4, 5, padding="same"), nn.BatchNorm2d(4), nn.ReLU(),
            nn.MaxPool2d(5, stride=4),                                   # 256 -> 63
            nn.Conv2d(4, 8, 5, padding="same"), nn.BatchNorm2d(8), nn.ReLU(),
            nn.MaxPool2d(4, stride=4),                                   # 63 -> 15
            nn.Conv2d(8, 16, 5, padding="same"), nn.BatchNorm2d(16), nn.ReLU(),
            nn.MaxPool2d(2, stride=4),                                   # 15 -> 4
            nn.Conv2d(16, 32, 2, padding="same"), nn.BatchNorm2d(32), nn.ReLU(),
        )
        self.classificador = nn.Sequential(nn.Flatten(), nn.Linear(32 * 4 * 4, classes))

    def forward(self, x):
        return self.classificador(self.features(x))


class ResNet18(nn.Module):
    """ResNet-18 pré-treinada no ImageNet, com a última camada trocada para 2 classes."""

    def __init__(self, classes=2):
        super().__init__()
        self.rede = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        self.rede.fc = nn.Linear(self.rede.fc.in_features, classes)
        self.register_buffer("media", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("desvio", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, x):
        return self.rede((x - self.media) / self.desvio)


def criar_modelo(nome):
    if nome == "cnn_artigo":
        return CNNArtigo()
    if nome == "resnet18":
        return ResNet18()
    raise ValueError(f"Modelo desconhecido: {nome}")

Overwriting /content/drive/MyDrive/tcc/src/modelos.py


In [8]:
%%writefile /content/drive/MyDrive/tcc/src/treino.py
import os
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import shutil


def fixar_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def salvar_seguro(obj, destino, tentativas=5):
    """Salva no disco do Colab e copia para o Drive, tentando de novo se o Drive falhar."""
    temporario = "/content/_checkpoint_tmp.pt"
    torch.save(obj, temporario)
    for t in range(tentativas):
        try:
            shutil.copy(temporario, destino + ".tmp")
            os.replace(destino + ".tmp", destino)
            return
        except OSError as e:
            print(f"  Falha ao salvar no Drive ({e}); tentando de novo em 10s...")
            time.sleep(10)
    raise RuntimeError(f"Não foi possível salvar {destino} depois de {tentativas} tentativas")


def preparar_lote(x, dispositivo, aumentar=False):
    """uint8 (N, H, W, 3) -> float (N, 3, H, W) entre 0 e 1, com flips aleatórios no treino."""
    x = x.to(dispositivo, non_blocking=True).permute(0, 3, 1, 2).float().div_(255)
    if aumentar:
        m = torch.rand(x.shape[0], device=dispositivo) < 0.5
        x[m] = x[m].flip(3)
        m = torch.rand(x.shape[0], device=dispositivo) < 0.5
        x[m] = x[m].flip(2)
    return x


@torch.no_grad()
def prever(modelo, X, dispositivo="cuda", tamanho_lote=128):
    """Probabilidade de cada crop ser IA."""
    modelo.eval()
    probs = []
    for i in range(0, len(X), tamanho_lote):
        xb = preparar_lote(X[i:i + tamanho_lote], dispositivo)
        with torch.autocast("cuda", dtype=torch.float16):
            saida = modelo(xb)
        probs.append(torch.softmax(saida.float(), 1)[:, 1].cpu())
    return torch.cat(probs).numpy()


@torch.no_grad()
def avaliar(modelo, X, y, dispositivo, criterio, tamanho_lote=128):
    modelo.eval()
    perda, acertos = 0.0, 0
    for i in range(0, len(X), tamanho_lote):
        xb = preparar_lote(X[i:i + tamanho_lote], dispositivo)
        yb = y[i:i + tamanho_lote].to(dispositivo)
        with torch.autocast("cuda", dtype=torch.float16):
            saida = modelo(xb)
        perda += criterio(saida.float(), yb).item() * len(yb)
        acertos += (saida.argmax(1) == yb).sum().item()
    return perda / len(X), acertos / len(X)


def treinar(modelo, X_tr, y_tr, X_val, y_val, pasta, lr, epocas, paciencia,
            tamanho_lote, seed, dispositivo="cuda"):
    os.makedirs(pasta, exist_ok=True)
    modelo.to(dispositivo)
    otimizador = torch.optim.AdamW(modelo.parameters(), lr=lr, weight_decay=1e-4)
    escalador = torch.amp.GradScaler("cuda")
    criterio = nn.CrossEntropyLoss()

    estado = {"epoca": 0, "melhor_perda": float("inf"), "sem_melhora": 0,
              "historico": [], "concluido": False}

    ultimo = f"{pasta}/ultimo.pt"
    if os.path.exists(ultimo):
        try:
            ck = torch.load(ultimo, map_location=dispositivo, weights_only=False)
        except Exception as e:
            print(f"Checkpoint corrompido ({e}); recomeçando este treino do zero.")
            os.remove(ultimo)
            ck = None
        if ck is not None:
            modelo.load_state_dict(ck["modelo"])
            otimizador.load_state_dict(ck["otimizador"])
            escalador.load_state_dict(ck["escalador"])
            estado = ck["estado"]
            if estado["concluido"]:
                print("Este treino já foi concluído.")
                return pd.DataFrame(estado["historico"])
            print(f"Retomando a partir da época {estado['epoca'] + 1}")

    n = len(X_tr)
    for epoca in range(estado["epoca"], epocas):
        inicio = time.time()
        modelo.train()
        ordem = torch.randperm(n, generator=torch.Generator().manual_seed(seed * 1000 + epoca))

        perda_total, acertos = 0.0, 0
        for i in range(0, n, tamanho_lote):
            idx = ordem[i:i + tamanho_lote]
            xb = preparar_lote(X_tr[idx], dispositivo, aumentar=True)
            yb = y_tr[idx].to(dispositivo)

            otimizador.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16):
                saida = modelo(xb)
                perda = criterio(saida, yb)
            escalador.scale(perda).backward()
            escalador.step(otimizador)
            escalador.update()

            perda_total += perda.item() * len(idx)
            acertos += (saida.argmax(1) == yb).sum().item()

        perda_val, acc_val = avaliar(modelo, X_val, y_val, dispositivo, criterio)
        estado["historico"].append({
            "epoca": epoca + 1,
            "perda_treino": perda_total / n, "acc_treino": acertos / n,
            "perda_val": perda_val, "acc_val": acc_val,
            "segundos": round(time.time() - inicio, 1),
        })
        estado["epoca"] = epoca + 1

        marca = ""
        if perda_val < estado["melhor_perda"]:
            estado["melhor_perda"] = perda_val
            estado["sem_melhora"] = 0
            salvar_seguro(modelo.state_dict(), f"{pasta}/melhor.pt")
            marca = "  <- melhor"
        else:
            estado["sem_melhora"] += 1

        parar_cedo = estado["sem_melhora"] >= paciencia
        estado["concluido"] = parar_cedo or estado["epoca"] >= epocas

        salvar_seguro({"modelo": modelo.state_dict(), "otimizador": otimizador.state_dict(),
                       "escalador": escalador.state_dict(), "estado": estado}, ultimo)
        pd.DataFrame(estado["historico"]).to_csv(f"{pasta}/historico.csv", index=False)

        h = estado["historico"][-1]
        print(f"Época {h['epoca']:2d} | treino: perda {h['perda_treino']:.4f} acc {h['acc_treino']:.3f}"
              f" | val: perda {h['perda_val']:.4f} acc {h['acc_val']:.3f} | {h['segundos']}s{marca}")

        if parar_cedo:
            print(f"Early stopping: {paciencia} épocas sem melhora na validação.")
            break

    return pd.DataFrame(estado["historico"])

Overwriting /content/drive/MyDrive/tcc/src/treino.py


In [9]:
from src import modelos, treino
importlib.reload(modelos); importlib.reload(treino)

MODELO      = "cnn_artigo"   # "cnn_artigo" ou "resnet18"
ENTRADA     = "ela"          # "ela" ou "rgb" (rgb = baseline sem ELA)
EXPERIMENTO = "png"          # "png", "jpeg90" ou "misto"
SEED        = 42             # usem 42, 7 e 123
AMOSTRA     = False          # True para testar o código; False para o treino de verdade

HIPER = {
    "cnn_artigo": {"lr": 1e-3, "epocas": 40, "paciencia": 6, "tamanho_lote": 64},
    "resnet18":   {"lr": 1e-4, "epocas": 20, "paciencia": 4, "tamanho_lote": 64},
}[MODELO].copy()
if AMOSTRA:
    HIPER["epocas"] = 2

NOME = f"{MODELO}_{ENTRADA}_{EXPERIMENTO}_seed{SEED}" + ("_amostra" if AMOSTRA else "")
PASTA = f"{config.MODELOS}/{NOME}"
print("Experimento:", NOME)
print("Hiperparâmetros:", HIPER)

Experimento: cnn_artigo_ela_png_seed42
Hiperparâmetros: {'lr': 0.001, 'epocas': 40, 'paciencia': 6, 'tamanho_lote': 64}


In [10]:
ORIGEM = f"{config.ELA_DIR}/{'etapa1_amostra' if AMOSTRA else 'etapa1'}"
VERSOES = {"png": ["original"], "jpeg90": ["jpeg90"], "misto": ["original", "jpeg90"]}[EXPERIMENTO]
os.makedirs("/content/dados", exist_ok=True)

def carregar(split):
    xs, ys = [], []
    for v in VERSOES:
        nome = f"{split}_{v}"
        local = f"/content/dados/{os.path.basename(ORIGEM)}_{nome}_{ENTRADA}.npy"
        if not os.path.exists(local):
            shutil.copy(f"{ORIGEM}/{nome}_{ENTRADA}.npy", local)
        xs.append(np.load(local))
        ys.append(pd.read_csv(f"{ORIGEM}/{nome}_rotulos.csv")["rotulo"].values)
    return torch.from_numpy(np.concatenate(xs)), torch.from_numpy(np.concatenate(ys)).long()

X_tr, y_tr = carregar("treino")
X_val, y_val = carregar("val")
print("Treino:", tuple(X_tr.shape), "| IA:", int(y_tr.sum()), "| reais:", int((y_tr == 0).sum()))
print("Val:   ", tuple(X_val.shape), "| IA:", int(y_val.sum()), "| reais:", int((y_val == 0).sum()))

Treino: (5600, 256, 256, 3) | IA: 2800 | reais: 2800
Val:    (1200, 256, 256, 3) | IA: 600 | reais: 600


In [11]:
treino.fixar_seed(SEED)
modelo = modelos.criar_modelo(MODELO)

os.makedirs(PASTA, exist_ok=True)
json.dump({"modelo": MODELO, "entrada": ENTRADA, "experimento": EXPERIMENTO,
           "seed": SEED, **HIPER}, open(f"{PASTA}/config.json", "w"), indent=2)

historico = treino.treinar(modelo, X_tr, y_tr, X_val, y_val, PASTA, seed=SEED, **HIPER)

Este treino já foi concluído.


In [12]:
import matplotlib.pyplot as plt

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.plot(historico["epoca"], historico["perda_treino"], label="treino")
a1.plot(historico["epoca"], historico["perda_val"], label="validação")
a1.set_xlabel("Época"); a1.set_ylabel("Perda"); a1.legend(); a1.set_title(NOME)
a2.plot(historico["epoca"], historico["acc_treino"], label="treino")
a2.plot(historico["epoca"], historico["acc_val"], label="validação")
a2.set_xlabel("Época"); a2.set_ylabel("Acurácia"); a2.legend()
plt.tight_layout()
plt.savefig(f"{PASTA}/curvas.png", dpi=150)
plt.show()

In [13]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from src import modelos, treino
import os
import time
import random
import shutil

HIPER_BASE = {
    "cnn_artigo": {"lr": 1e-3, "epocas": 40, "paciencia": 6, "tamanho_lote": 64},
    "resnet18":   {"lr": 1e-4, "epocas": 20, "paciencia": 4, "tamanho_lote": 64},
}
VERSOES_EXP = {"png": ["original"], "jpeg90": ["jpeg90"], "misto": ["original", "jpeg90"]}
ORIGEM = f"{config.ELA_DIR}/etapa1"

def carregar_conjunto(split, versoes, entrada):
    os.makedirs("/content/dados", exist_ok=True)
    xs, ys = [], []
    for v in versoes:
        nome = f"{split}_{v}"
        local = f"/content/dados/etapa1_{nome}_{entrada}.npy"
        if not os.path.exists(local):
            shutil.copy(f"{ORIGEM}/{nome}_{entrada}.npy", local)
        xs.append(np.load(local))
        ys.append(pd.read_csv(f"{ORIGEM}/{nome}_rotulos.csv")["rotulo"].values)
    return torch.from_numpy(np.concatenate(xs)), torch.from_numpy(np.concatenate(ys)).long()

# CONFIGS = [
#     (modelo, entrada, exp, seed)
#     for seed in [42, 7, 123]
#     for exp in ["png", "jpeg90", "misto"]
#     for modelo, entrada in [("cnn_artigo", "ela"), ("cnn_artigo", "rgb"), ("resnet18", "ela")]
# ]

CONFIGS = [
    ("resnet18", "rgb", exp, seed)
    for seed in [42, 7, 123]
    for exp in ["png", "jpeg90", "misto"]
]

for i, (MODELO, ENTRADA, EXPERIMENTO, SEED) in enumerate(CONFIGS, 1):
    NOME = f"{MODELO}_{ENTRADA}_{EXPERIMENTO}_seed{SEED}"
    PASTA = f"{config.MODELOS}/{NOME}"
    print(f"\n===== [{i}/{len(CONFIGS)}] {NOME} =====")

    HIPER = HIPER_BASE[MODELO].copy()
    versoes = VERSOES_EXP[EXPERIMENTO]
    X_tr, y_tr = carregar_conjunto("treino", versoes, ENTRADA)
    X_val, y_val = carregar_conjunto("val", versoes, ENTRADA)

    treino.fixar_seed(SEED)
    modelo = modelos.criar_modelo(MODELO)
    os.makedirs(PASTA, exist_ok=True)
    json.dump({"modelo": MODELO, "entrada": ENTRADA, "experimento": EXPERIMENTO,
               "seed": SEED, **HIPER}, open(f"{PASTA}/config.json", "w"), indent=2)

    hist = treino.treinar(modelo, X_tr, y_tr, X_val, y_val, PASTA, seed=SEED, **HIPER)

    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
    for col, eixo, rotulo in [("perda", a1, "Perda"), ("acc", a2, "Acurácia")]:
        eixo.plot(hist["epoca"], hist[f"{col}_treino"], label="treino")
        eixo.plot(hist["epoca"], hist[f"{col}_val"], label="validação")
        eixo.set_xlabel("Época"); eixo.set_ylabel(rotulo); eixo.legend()
    a1.set_title(NOME)
    plt.tight_layout(); plt.savefig(f"{PASTA}/curvas.png", dpi=150); plt.close(fig)

    del modelo, X_tr, y_tr, X_val, y_val
    torch.cuda.empty_cache()

print("\nTodas as configurações concluídas.")


===== [1/9] resnet18_rgb_png_seed42 =====
Época  1 | treino: perda 0.0810 acc 0.968 | val: perda 0.0895 acc 0.968 | 9.6s  <- melhor
Época  2 | treino: perda 0.0138 acc 0.995 | val: perda 0.0929 acc 0.972 | 10.1s
Época  3 | treino: perda 0.0099 acc 0.996 | val: perda 0.0723 acc 0.979 | 9.6s  <- melhor
Época  4 | treino: perda 0.0115 acc 0.996 | val: perda 0.0813 acc 0.975 | 10.1s
Época  5 | treino: perda 0.0047 acc 0.999 | val: perda 0.0724 acc 0.981 | 9.8s
Época  6 | treino: perda 0.0212 acc 0.993 | val: perda 0.0562 acc 0.981 | 9.5s  <- melhor
Época  7 | treino: perda 0.0190 acc 0.995 | val: perda 0.0435 acc 0.983 | 9.8s  <- melhor
Época  8 | treino: perda 0.0038 acc 0.999 | val: perda 0.0754 acc 0.983 | 9.8s
Época  9 | treino: perda 0.0070 acc 0.999 | val: perda 0.0619 acc 0.986 | 9.9s
Época 10 | treino: perda 0.0036 acc 0.999 | val: perda 0.0634 acc 0.983 | 9.6s
Época 11 | treino: perda 0.0198 acc 0.993 | val: perda 0.0854 acc 0.975 | 9.6s
Early stopping: 4 épocas sem melhora na va